In [0]:
# Set the client code - change this to match your data
clientCode = "TestClient"

# Paths where Gold tables will be created
goldMemPerBrdgPath = f"/Volumes/pharma_catalog/Gold/MA/Client/MemberPersonBridge/{clientCode}"
goldMemPath = f"/Volumes/pharma_catalog/Gold/MA/Client/Member/{clientCode}"

print(f"Client Code: {clientCode}")
print(f"MemberPersonBridge output: {goldMemPerBrdgPath}")
print(f"Member output: {goldMemPath}")

In [0]:
print("\n" + "="*60)
print("STEP 1: Running MemberPersonBridge notebook...")
print("="*60 + "\n")

try:
    result = dbutils.notebook.run(
        "./databricks-dev/Notebooks/GoldLayerProcessing/MemberPersonBridge",
        timeout_seconds=10,  # 30 minutes
        arguments={"ClientContainer": clientCode}
    )
    print("✓ MemberPersonBridge completed successfully")
    print(f"Result: {result}")
except Exception as e:
    print(f"✗ MemberPersonBridge failed: {str(e)}")
    raise

In [0]:
print("\n" + "="*60)
print("STEP 2: Running Member notebook...")
print("="*60 + "\n")

try:
    result = dbutils.notebook.run(
        "./databricks-dev/Notebooks/GoldLayerProcessing/Member",
        timeout_seconds=1800,  # 30 minutes
        arguments={"ClientContainer": clientCode}
    )
    print("✓ Member completed successfully")
    print(f"Result: {result}")
except Exception as e:
    print(f"✗ Member failed: {str(e)}")
    raise

In [0]:
print("\n" + "="*60)
print("STEP 3: Verifying Gold Layer Delta Tables")
print("="*60 + "\n")

def verify_delta_table(path, table_name):
    try:
        # Check if path exists
        files = dbutils.fs.ls(path)
        print(f"✓ {table_name} path exists: {path}")
        
        # Read the Delta table
        df = spark.read.format("delta").load(path)
        row_count = df.count()
        col_count = len(df.columns)
        
        print(f"  - Row count: {row_count:,}")
        print(f"  - Column count: {col_count}")
        print(f"  - Columns: {', '.join(df.columns[:10])}{'...' if col_count > 10 else ''}")
        
        if row_count > 0:
            print(f"  - Sample data (first 3 rows):")
            df.show(3, truncate=True)
        
        return True
    except Exception as e:
        print(f"✗ {table_name} verification failed: {str(e)}")
        return False

# Verify both tables
bridge_ok = verify_delta_table(goldMemPerBrdgPath, "MemberPersonBridge")
print()
member_ok = verify_delta_table(goldMemPath, "Member")

print("\n" + "="*60)
if bridge_ok and member_ok:
    print("✓✓✓ SUCCESS: Both Gold Layer tables created successfully!")
else:
    print("✗✗✗ FAILURE: Some tables were not created properly")
print("="*60)